1. Imports + ENV

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

# notebook je u: api/notebooks
load_dotenv(Path("../.env"))           # api/.env (OPENAI_*, QDRANT_* ako su tu)
load_dotenv(Path("../../.env.local"))  # root .env.local (NEXT_PUBLIC_API_URL)

print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_EMBED_MODEL:", os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small"))

print("QDRANT_URL:", os.getenv("QDRANT_URL"))
print("QDRANT_API_KEY present:", bool(os.getenv("QDRANT_API_KEY")))

print("NEXT_PUBLIC_API_URL:", os.getenv("NEXT_PUBLIC_API_URL"))

OPENAI_API_KEY present: True
OPENAI_EMBED_MODEL: text-embedding-3-small
QDRANT_URL: https://f5bc9fb3-c963-437a-b5e0-36ad9101698f.eu-central-1-0.aws.cloud.qdrant.io
QDRANT_API_KEY present: True
NEXT_PUBLIC_API_URL: http://127.0.0.1:8000


2. Config

In [4]:
from pathlib import Path

DATA_DIR = Path("../tv_data")
CSV_FILENAME = "Monthly iptv - channel monthly rating.csv"

COLLECTION_NAME = "tv_monthly_channel_rating"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

print("DATA_DIR:", DATA_DIR.resolve(), "exists:", DATA_DIR.exists())
print("CSV file:", (DATA_DIR / CSV_FILENAME).resolve(), "exists:", (DATA_DIR / CSV_FILENAME).exists())
print("COLLECTION_NAME:", COLLECTION_NAME)
print("CHUNK_SIZE:", CHUNK_SIZE, "CHUNK_OVERLAP:", CHUNK_OVERLAP)

DATA_DIR: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data exists: True
CSV file: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data\Monthly iptv - channel monthly rating.csv exists: True
COLLECTION_NAME: tv_monthly_channel_rating
CHUNK_SIZE: 1000 CHUNK_OVERLAP: 200


3. Load CSV -> TV data

In [5]:
import pandas as pd
from langchain_core.documents import Document

CSV_PATH = DATA_DIR / CSV_FILENAME

df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8", dtype=str)
print("rows:", len(df))
print("columns:", list(df.columns))

docs = []
for i, row in df.iterrows():
    text = " | ".join(
        [f"{k}={v}" for k, v in row.items() if pd.notna(v) and str(v).strip() != ""]
    )
    if text.strip():
        docs.append(Document(page_content=text, metadata={"row": int(i), "file": CSV_PATH.name}))

print(f"Loaded {len(docs)} documents")
print("Sample doc:", docs[0].page_content[:250])
print("Sample meta:", docs[0].metadata)

c:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


rows: 138642
columns: ['month_partition', 'watch_date', 'channelname', 'content_playback_type', 'daily_total_minute', 'nr_unique_viewers']
Loaded 138642 documents
Sample doc: month_partition=202601 | watch_date=2026-01-01 | channelname=BBC Earth | content_playback_type=LTV | daily_total_minute=163974,13 | nr_unique_viewers=4413
Sample meta: {'row': 0, 'file': 'Monthly iptv - channel monthly rating.csv'}


4. Split into Chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")
print("Sample chunk:", chunks[0].page_content[:250])
print("Sample meta:", chunks[0].metadata)

Split into 138642 chunks
Sample chunk: month_partition=202601 | watch_date=2026-01-01 | channelname=BBC Earth | content_playback_type=LTV | daily_total_minute=163974,13 | nr_unique_viewers=4413
Sample meta: {'row': 0, 'file': 'Monthly iptv - channel monthly rating.csv'}


5. Qdrant Cloud connection test

In [7]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

print(client.get_collections())

collections=[]


6. Create collection + Ingest

In [8]:
from langchain_openai import OpenAIEmbeddings
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

# clean start (da ne dupliras)
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

# create collection (1536 za text-embedding-3-small)
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

embeddings = OpenAIEmbeddings(
    model=os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

BATCH = 500
total = len(chunks)
count = 0

for i in range(0, total, BATCH):
    batch = chunks[i:i+BATCH]
    ids = vector_store.add_documents(batch)
    count += len(ids)

    if (i // BATCH) % 10 == 0:
        print(f"Ingested {count}/{total}")

print(f"DONE: Ingested {count} chunks into '{COLLECTION_NAME}' (Qdrant Cloud)")

Ingested 500/138642
Ingested 5500/138642
Ingested 10500/138642
Ingested 15500/138642
Ingested 20500/138642
Ingested 25500/138642
Ingested 30500/138642
Ingested 35500/138642
Ingested 40500/138642
Ingested 45500/138642
Ingested 50500/138642
Ingested 55500/138642
Ingested 60500/138642
Ingested 65500/138642
Ingested 70500/138642
Ingested 75500/138642
Ingested 80500/138642
Ingested 85500/138642
Ingested 90500/138642
Ingested 95500/138642
Ingested 100500/138642
Ingested 105500/138642
Ingested 110500/138642
Ingested 115500/138642
Ingested 120500/138642
Ingested 125500/138642
Ingested 130500/138642
Ingested 135500/138642
DONE: Ingested 138642 chunks into 'tv_monthly_channel_rating' (Qdrant Cloud)


7. point_count

In [9]:
info = client.get_collection(COLLECTION_NAME)
print("points_count:", info.points_count)

points_count: 138642


8. Retrival test

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

hits = retriever.invoke("BBC Earth unique viewers")
for h in hits:
    print("-", h.page_content)

- month_partition=202512 | watch_date=2025-12-08 | channelname=BBC Earth | content_playback_type=LTV | daily_total_minute=198947,5 | nr_unique_viewers=4659
- month_partition=202502 | watch_date=2025-02-23 | channelname=BBC Earth | content_playback_type=IR | daily_total_minute=13239,9 | nr_unique_viewers=198
- month_partition=202509 | watch_date=2025-09-09 | channelname=BBC Earth | content_playback_type=IR | daily_total_minute=4212,46 | nr_unique_viewers=106
- month_partition=202512 | watch_date=2025-12-28 | channelname=BBC Earth | content_playback_type=LTV | daily_total_minute=329845,09 | nr_unique_viewers=6737
- month_partition=202506 | watch_date=2025-06-18 | channelname=BBC Earth | content_playback_type=LTV | daily_total_minute=262861,15 | nr_unique_viewers=4287
